In [ ]:
import pandas as pd
import numpy as np

ZERO_SHOT_PATH = "/content/drive/MyDrive/Surgical-VLM/results/ZeroShot Results/visionllm_zeroshot_predictions.csv"
FINE_TUNED_PATH = "/content/drive/MyDrive/Surgical-VLM/results/VLM Results/visionllm_predictions.csv"

zero_shot_df = pd.read_csv(ZERO_SHOT_PATH)
fine_tuned_df = pd.read_csv(FINE_TUNED_PATH)

print("Zero-shot:", zero_shot_df.shape)
print("Fine-tuned:", fine_tuned_df.shape)

Zero-shot: (5600, 5)
Fine-tuned: (5600, 5)


In [ ]:
CANONICAL_LABELS = {

    "Instrument Recognition": [
        "grasper",
        "bipolar",
        "hook",
        "scissors",
        "clipper",
        "irrigator",
    ],

    "Action Recognition": [
        "grasp",
        "retract",
        "dissect",
        "coagulate",
        "clip",
        "cut",
        "aspirate",
        "irrigate",
        "pack",
        "null_verb",
    ],

    "Tissue and Organ Recognition": [
        "gallbladder",
        "cystic artery",
        "cystic duct",
        "cystic plate",
        "liver",
        "specimen bag",
        "fluid",
        "abdominal wall cavity",
        "omentum",
        "blood_vessel",
        "gut",
        "peritoneum",
        "cystic_pedicle",
        "adhesion",
        "null_target"
    ],

    "Phase Recognition": [
        "Preparation",
        "Calot Triangle Dissection",
        "Clipping Cutting",
        "Gallbladder Dissection",
        "Gallbladder Packaging",
        "Cleaning Coagulation",
        "Gallbladder Retraction",
    ],
}

In [ ]:
def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = text.replace("_", " ")

    text = " ".join(text.split())

    return text


def extract_labels(text, task):

    text = normalize_text(text)

    labels = CANONICAL_LABELS[task]

    found = []

    # Longest first
    labels_sorted = sorted(
        labels,
        key=lambda x: len(normalize_text(x)),
        reverse=True
    )

    for label in labels_sorted:

        normalized_label = normalize_text(label)

        if normalized_label in text:
            found.append(label)

    # Deduplicate while preserving order
    found = list(dict.fromkeys(found))

    return found

In [ ]:
fine_tuned_df = fine_tuned_df[
    fine_tuned_df["task"].isin(
        CANONICAL_LABELS.keys()
    )
].copy()

fine_tuned_df["gt_labels"] = fine_tuned_df.apply(
    lambda row: extract_labels(
        row["ground_truth"],
        row["task"]
    ),
    axis=1
)

display(
    fine_tuned_df[
        ["task", "ground_truth", "gt_labels"]
    ].head(20)
)

,task,ground_truth,gt_labels
0,Action Recognition,The grasper is performing a retract action in ...,"[retract, grasp]"
1,Action Recognition,"retract, dissect","[retract, dissect]"
2,Action Recognition,The grasper is performing a retract action in ...,"[retract, grasp]"
3,Action Recognition,"retract, dissect","[retract, dissect]"
4,Action Recognition,dissect,[dissect]
5,Action Recognition,The bipolar is involved in performing coagulat...,[coagulate]
6,Action Recognition,The hook completes a dissect task in this surg...,[dissect]
7,Action Recognition,The grasper performs a retract action during t...,"[retract, grasp]"
8,Action Recognition,The grasper is assigned the task of retract in...,"[retract, grasp]"
9,Action Recognition,The grasper is accomplishing a retract task in...,"[retract, grasp]"


In [ ]:
from collections import Counter

majority_labels = {}

for task in CANONICAL_LABELS:

    subset = fine_tuned_df[
        fine_tuned_df["task"] == task
    ]

    counter = Counter()

    for labels in subset["gt_labels"]:

        for label in labels:
            counter[label] += 1

    majority_label, majority_count = (
        counter.most_common(1)[0]
    )

    majority_labels[task] = majority_label

    print(
        f"{task}: "
        f"{majority_label} "
        f"({majority_count} occurrences)"
    )

Instrument Recognition: hook (557 occurrences)
Action Recognition: dissect (407 occurrences)
Tissue and Organ Recognition: gallbladder (657 occurrences)
Phase Recognition: Gallbladder Dissection (319 occurrences)


In [ ]:
fine_tuned_df["majority_prediction"] = (
    fine_tuned_df["task"]
    .map(majority_labels)
)

In [ ]:
def example_metrics(
    predicted,
    ground_truth
):

    predicted = set(predicted)
    ground_truth = set(ground_truth)

    tp = len(
        predicted & ground_truth
    )

    fp = len(
        predicted - ground_truth
    )

    fn = len(
        ground_truth - predicted
    )

    return tp, fp, fn

In [ ]:
baseline_rows = []

for task in CANONICAL_LABELS:

    subset = fine_tuned_df[
        fine_tuned_df["task"] == task
    ]

    majority_label = majority_labels[task]

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for _, row in subset.iterrows():

        predicted = [
            majority_label
        ]

        ground_truth = row["gt_labels"]

        tp, fp, fn = example_metrics(
            predicted,
            ground_truth
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = (
        total_tp /
        (total_tp + total_fp)
        if (total_tp + total_fp) > 0
        else 0
    )

    recall = (
        total_tp /
        (total_tp + total_fn)
        if (total_tp + total_fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    baseline_rows.append({

        "Task": task,

        "Majority label":
            majority_label,

        "N":
            len(subset),

        "Micro precision":
            precision,

        "Micro recall":
            recall,

        "Micro F1":
            f1,
    })

majority_multilabel = pd.DataFrame(
    baseline_rows
)

display(
    majority_multilabel
)

,Task,Majority label,N,Micro precision,Micro recall,Micro F1
0,Instrument Recognition,hook,800,0.69625,0.481417,0.569239
1,Action Recognition,dissect,800,0.50875,0.352992,0.416795
2,Tissue and Organ Recognition,gallbladder,800,0.82125,0.686520,0.747866
3,Phase Recognition,Gallbladder Dissection,800,0.39875,0.398750,0.398750


In [ ]:
exact_rows = []

for task in CANONICAL_LABELS:

    subset = fine_tuned_df[
        fine_tuned_df["task"] == task
    ]

    majority_label = majority_labels[task]

    exact_correct = 0

    for _, row in subset.iterrows():

        predicted = {
            majority_label
        }

        ground_truth = set(
            row["gt_labels"]
        )

        if predicted == ground_truth:
            exact_correct += 1

    exact_accuracy = (
        exact_correct /
        len(subset)
    )

    exact_rows.append({

        "Task": task,

        "Majority exact-set accuracy":
            exact_accuracy
    })

majority_exact = pd.DataFrame(
    exact_rows
)

In [ ]:
majority_baseline_table = (
    majority_multilabel
    .merge(
        majority_exact,
        on="Task"
    )
)

display(
    majority_baseline_table
)

majority_baseline_table.to_csv(
    "/content/drive/MyDrive/Surgical-VLM/results/majority_baseline_table.csv",
    index=False
)

,Task,Majority label,N,Micro precision,Micro recall,Micro F1,Majority exact-set accuracy
0,Instrument Recognition,hook,800,0.69625,0.481417,0.569239,0.31250
1,Action Recognition,dissect,800,0.50875,0.352992,0.416795,0.34375
2,Tissue and Organ Recognition,gallbladder,800,0.82125,0.686520,0.747866,0.63250
3,Phase Recognition,Gallbladder Dissection,800,0.39875,0.398750,0.398750,0.39875
